# Spray Chart Diffusion — Interactive Demo

Load a trained model, pick a batter and game situation, generate and plot the predicted spray chart distribution.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path('spray_chart_diffusion')))

import json
import numpy as np
import torch
import matplotlib.pyplot as plt

from spray_chart_diffusion.src.inference.sample import load_model, generate
from spray_chart_diffusion.src.inference.inpaint import inpaint, build_partial_chart
from spray_chart_diffusion.src.analysis.visualize import plot_spray_chart
from spray_chart_diffusion.src.data.preprocess import SITUATION_CODES

In [ ]:
# --- Configuration ---
CHECKPOINT    = 'spray_chart_diffusion/checkpoints/best.pt'
PROCESSED_DIR = 'spray_chart_diffusion/data/processed'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

model = load_model(CHECKPOINT, PROCESSED_DIR, device=device, inpaint_mode=True)
print('Model loaded.')

with open(f'{PROCESSED_DIR}/batter_id_map.json') as f:
    batter_id_map = {int(k): v for k, v in json.load(f).items()}

print('Available situation codes:')
for code in SITUATION_CODES:
    print(' ', code)

In [ ]:
# --- User inputs ---
# Replace BATTER_MLBAM with any MLBAM batter ID present in the training data.
# Example: Mike Trout = 545361, Shohei Ohtani = 660271

BATTER_MLBAM  = 545361          # MLBAM batter ID
SITUATION     = 'ahead_R_fastball'   # one of SITUATION_CODES, or 'full'
NUM_SAMPLES   = 30
GUIDANCE      = 3.0
INFER_STEPS   = 200

batter_idx = batter_id_map.get(BATTER_MLBAM)
if batter_idx is None:
    print(f'Batter {BATTER_MLBAM} not found in id map. Using index 0.')
    batter_idx = 0
print(f'Batter MLBAM {BATTER_MLBAM} → embed row {batter_idx}')

In [ ]:
# --- Generate spray chart ---
samples = generate(
    model, batter_idx,
    situation_str=SITUATION,
    num_samples=NUM_SAMPLES,
    guidance_scale=GUIDANCE,
    num_inference_steps=INFER_STEPS,
    device=device,
)
mean_chart = samples.mean(axis=0)
std_chart  = samples.std(axis=0)
print(f'Generated {NUM_SAMPLES} samples. Shape: {samples.shape}')

In [ ]:
# --- Plot mean and uncertainty ---
fig, axes = plt.subplots(1, 3, figsize=(12, 4))

plot_spray_chart(mean_chart, ax=axes[0],
                 title=f'Mean density\nMLBAM {BATTER_MLBAM} | {SITUATION}')

plot_spray_chart(std_chart, ax=axes[1],
                 title='Uncertainty (std across samples)',
                 cmap='Blues')

# One individual sample
plot_spray_chart(samples[0], ax=axes[2],
                 title='Single sample draw')

fig.suptitle(f'Spray Chart Diffusion  —  MLBAM {BATTER_MLBAM}  |  {SITUATION}', y=1.02)
fig.tight_layout()
plt.show()

In [ ]:
# --- Optional: inpainting from k observed events ---
# Supply raw hc_x / hc_y arrays from real Statcast data, or use synthetic ones.

import pandas as pd
RAW_CSV = f'spray_chart_diffusion/data/raw/statcast_2023.csv'
try:
    raw = pd.read_csv(RAW_CSV, low_memory=False)
    batter_raw = raw[raw['batter'] == BATTER_MLBAM].dropna(subset=['hc_x','hc_y'])
    hx, hy = batter_raw['hc_x'].values, batter_raw['hc_y'].values
    print(f'Found {len(hx)} raw 2023 events for batter {BATTER_MLBAM}')
except FileNotFoundError:
    # Synthetic fallback
    rng = np.random.default_rng(0)
    hx = rng.normal(125, 30, 200)
    hy = rng.normal(175, 25, 200)
    print('Using synthetic events (raw CSV not found).')

K = 20   # number of observed events to condition on
inpaint_samples = inpaint(
    model, batter_idx, hx, hy, k=K,
    situation_str=SITUATION,
    num_samples=30,
    num_inference_steps=INFER_STEPS,
    device=device,
)

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
from spray_chart_diffusion.src.inference.inpaint import build_partial_chart
partial = build_partial_chart(hx, hy, k=K)
plot_spray_chart(partial, ax=axes[0], title=f'Partial chart (k={K} events)')
plot_spray_chart(inpaint_samples.mean(axis=0), ax=axes[1],
                 title='Inpainting mean prediction')
fig.tight_layout()
plt.show()